# Early Dementia Detection from Brain MRI
### YOLOv8 Classification Benchmark
**Author:** Claude (Adaptive AI Collaborator)
**Methodology:**
* **Data:** OASIS MRI Dataset, same patient-level splits as the original benchmark run.
* **Architecture:** YOLOv8n-cls, fine-tuned from COCO-pretrained classification weights.
* **Robustness:** Reuses `src/data_pipeline.py` (stratified per-category patient split) and `src/evaluate.py` so the CNN and YOLOv8 are benchmarked on identical patients with identical metric code.
* **Goal:** Reproduce the documented **91.64% accuracy / 0.7869 Scott's Pi / 0.8447 QWK** for YOLOv8 (and 75.73%/0.4257/0.6680 for the CNN), and only replace `models/best.pt` after a side-by-side comparison is approved.


## 1. Setup
Install `ultralytics` (which bundles the YOLOv8 classification trainer) and import the libraries used throughout this notebook, including TensorFlow so we can load the existing CNN for a direct comparison.


In [ ]:
%pip install -q ultralytics

import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from ultralytics import YOLO

print("Ultralytics + torch ready. CUDA available:", torch.cuda.is_available())


## 2. Load Metadata & Patient-Level Splits
Rather than re-implementing metadata extraction and splitting, we import the exact same functions used by the CNN pipeline from **`src/data_pipeline.py`**. The split is **stratified per category**: most classes get a normal 80/10/10-style patient split, but `Moderate Dementia` has only 2 patients in the whole dataset, so both patients are used in all three splits — otherwise a pooled split would leave 0 test patients for that class. This guarantees every split contains all 4 classes and matches the patients used in the original benchmark run.


In [ ]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(REPO_ROOT / "src"))

from data_pipeline import CATEGORIES, create_metadata_df, patient_level_split, hybrid_resample
from evaluate import evaluate_predictions

CLASS_NAMES = list(CATEGORIES.keys())

# Update this to wherever the OASIS `Data/<category>/...jpg` folders live on disk
DATA_PATH = REPO_ROOT / "dataset" / "Data"

df = create_metadata_df(DATA_PATH)
train_df, val_df, test_df = patient_level_split(df)

print(f"Total scans: {len(df)} | Unique patients: {df['patient_id'].nunique()}")
for name, split_df in [("Training", train_df), ("Validation", val_df), ("Testing", test_df)]:
    print(f"{name:11s}: {split_df['patient_id'].nunique()} patients | {len(split_df)} scans | classes: {sorted(split_df['category'].unique())}")

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert len(split_df["category"].unique()) == 4, f"{name} split is missing a class!"
print("\nAll 4 classes confirmed present in every split.")


## 3. Hybrid Resampling (Training Split Only)
Same **137:1 imbalance** problem as the CNN, so we reuse `hybrid_resample` from `src/data_pipeline.py` with the same `target_samples_per_class=8000`. Validation and test stay at their natural distribution — resampling only the training split keeps the test-set accuracy an honest estimate of real-world performance.


In [ ]:
train_df_balanced = hybrid_resample(train_df, target_samples_per_class=8000)

print("Balanced training distribution:")
print(train_df_balanced["category"].value_counts())


## 4. Organize Images into YOLO's Classification Folder Structure
YOLOv8 classification mode expects **ImageFolder-style directories** (`<root>/train/<class>/*.jpg`, `<root>/val/<class>/*.jpg`, `<root>/test/<class>/*.jpg`), not a dataframe. `organize_into_yolo_folders` below materializes our three splits into that layout under `yolo_data/` (already listed in `.gitignore`, since it's a derived cache, not source data).

We use symlinks where the OS allows it (Linux/Colab) to avoid duplicating tens of thousands of images on disk, falling back to a real copy only where symlinks aren't permitted (e.g. Windows without Developer Mode). The function is idempotent — re-running it skips files that already exist.


In [ ]:
def organize_into_yolo_folders(splits, output_root):
    output_root = Path(output_root)
    for split_name, split_df in splits.items():
        for i, row in enumerate(tqdm(split_df.itertuples(), total=len(split_df), desc=f"Organizing {split_name}")):
            dest_dir = output_root / split_name / row.category
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest_path = dest_dir / f"{i}_{os.path.basename(row.path)}"
            if dest_path.exists():
                continue
            try:
                os.symlink(os.path.abspath(row.path), dest_path)
            except OSError:
                shutil.copy2(row.path, dest_path)

YOLO_DATA_ROOT = REPO_ROOT / "yolo_data"
organize_into_yolo_folders(
    {"train": train_df_balanced, "val": val_df, "test": test_df},
    YOLO_DATA_ROOT,
)

print("YOLO folder structure ready at:", YOLO_DATA_ROOT)


## 5. Load a Pretrained YOLOv8 Classification Checkpoint
We start from `yolov8n-cls.pt` — pretrained on ImageNet — the same nano-sized checkpoint already documented in `app.py` (1.44M params). Ultralytics downloads it automatically on first use.


In [ ]:
model = YOLO("yolov8n-cls.pt")


## 6. Fine-Tune on OASIS
Hyperparameters match the original benchmark run:
* **`epochs=30`**, **`patience=10`** — YOLOv8's built-in early stopping: training stops if fitness hasn't improved for 10 consecutive epochs, and the best epoch's weights are kept.
* **`optimizer='Adam', lr0=0.001`** — matches the original run rather than Ultralytics' default SGD schedule.
* **`imgsz=128`** — matches the CNN's `IMG_SIZE=(128, 128)` and the `imgsz=128` already used for YOLO inference in `app.py`.

Weights are written under `runs/classify/oasis_yolo_candidate/weights/`, **not** `models/best.pt` — we don't touch the incumbent checkpoint until the comparison in Section 11 is approved.


In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"

train_results = model.train(
    data=str(YOLO_DATA_ROOT),
    task="classify",
    epochs=30,
    patience=10,
    imgsz=128,
    batch=32,
    optimizer="Adam",
    lr0=0.001,
    seed=42,
    device=device,
    project=str(REPO_ROOT / "runs" / "classify"),
    name="oasis_yolo_candidate",
    exist_ok=True,
    pretrained=True,
)


## 7. Candidate Weights — Saved Separately, Not Overwriting `models/best.pt`
The best checkpoint from this run is copied to `models/yolo_candidate.pt` — a separate path from the incumbent `models/best.pt`. We only decide whether to promote it after evaluating both side by side (Section 11).


In [ ]:
candidate_source = Path(train_results.save_dir) / "weights" / "best.pt"
CANDIDATE_PATH = REPO_ROOT / "models" / "yolo_candidate.pt"

shutil.copy2(candidate_source, CANDIDATE_PATH)
print("Candidate weights saved to:", CANDIDATE_PATH)


## 8. Load the Saved CNN Model
To reproduce the documented **dual-model benchmark** (not just the YOLO half), we also load the custom 6-block CNN from `models/dementia_detection_model_final.h5`. Both models will run on the exact same `test_df`.


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

CNN_MODEL_PATH = REPO_ROOT / "models" / "dementia_detection_model_final.h5"

cnn_model = load_model(CNN_MODEL_PATH)
print("CNN model loaded")
print(f"  Input shape : {cnn_model.input_shape}")
print(f"  Output shape: {cnn_model.output_shape}")


## 9. Evaluate on the Test Split
First, Ultralytics' built-in `model.val()` gives Top-1 / Top-5 accuracy on the held-out test patients directly from the folder structure.


In [ ]:
candidate_model = YOLO(CANDIDATE_PATH)

val_metrics = candidate_model.val(data=str(YOLO_DATA_ROOT), split="test", imgsz=128)
print(f"Ultralytics Top-1 accuracy: {val_metrics.top1 * 100:.2f}%")
print(f"Ultralytics Top-5 accuracy: {val_metrics.top5 * 100:.2f}%")


## 10. Run Both Models on the Same Test Set, Manually Score with `src/evaluate.py`
The documented benchmark (91.64%/0.7869/0.8447 for YOLO, 75.73%/0.4257/0.6680 for the CNN) uses the manual metric functions in `src/evaluate.py`, not Ultralytics' built-in metrics. We predict with both models on the raw test dataframe (not the copied folder) so we keep the ordinal `label` column.

For YOLO, we map its alphabetically-sorted class indices back to our `CATEGORIES` ordinal encoding via `model.names`. For the CNN, passing `classes=CLASS_NAMES` (in `CATEGORIES`' insertion order) to `flow_from_dataframe` keeps its output indices aligned with `label` directly, so no remapping is needed there.


In [ ]:
IMG_SIZE = (128, 128)

# ── CNN predictions ──────────────────────────────────────────────────────
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="path",
    y_col="category",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode="sparse",
    shuffle=False,
)

cnn_probs = cnn_model.predict(test_generator, verbose=1)
cnn_preds = np.argmax(cnn_probs, axis=1)
y_true = np.array(test_generator.classes).astype(int)

# ── YOLO predictions ─────────────────────────────────────────────────────
def predict_and_score(yolo_model, eval_df):
    preds = []
    for result in yolo_model.predict(eval_df["path"].tolist(), imgsz=128, batch=32, stream=True, verbose=False):
        class_name = yolo_model.names[int(result.probs.top1)]
        preds.append(CATEGORIES[class_name])
    return np.array(preds)

yolo_preds = predict_and_score(candidate_model, test_df)

print(f"CNN predictions : {len(cnn_preds)}")
print(f"YOLO predictions: {len(yolo_preds)}")


## 11. Benchmark Results & Comparison to Documented Targets


In [ ]:
cnn_metrics = evaluate_predictions(y_true, cnn_preds, num_classes=4)
yolo_metrics = evaluate_predictions(y_true, yolo_preds, num_classes=4)

agreement = float(np.mean(cnn_preds == yolo_preds))
inter_model_qwk = evaluate_predictions(cnn_preds, yolo_preds, num_classes=4)["qwk"]

summary = pd.DataFrame([
    {"Model": "Custom 6-Block CNN", **cnn_metrics},
    {"Model": "YOLOv8", **yolo_metrics},
])
print(summary.to_string(index=False))
print(f"\nPrediction Agreement : {agreement:.4f} ({agreement * 100:.2f}%)")
print(f"Inter-model QWK      : {inter_model_qwk:.4f}")

DOCUMENTED_TARGETS = {
    "Custom 6-Block CNN": {"accuracy": 0.7573, "scotts_pi": 0.4257, "qwk": 0.6680},
    "YOLOv8": {"accuracy": 0.9164, "scotts_pi": 0.7869, "qwk": 0.8447},
}

print("\nDelta vs documented target:")
for model_name, metrics in [("Custom 6-Block CNN", cnn_metrics), ("YOLOv8", yolo_metrics)]:
    target = DOCUMENTED_TARGETS[model_name]
    print(f"  {model_name}:")
    for key in ("accuracy", "scotts_pi", "qwk"):
        print(f"    {key:10s}: {metrics[key]:.4f}  (target {target[key]:.4f}, delta {metrics[key] - target[key]:+.4f})")


## 12. Confusion Matrices & Classification Reports


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

cnn_cm = confusion_matrix(y_true, cnn_preds)
yolo_cm = confusion_matrix(y_true, yolo_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
short_names = ["Non\nDemented", "Very Mild", "Mild", "Moderate"]
for ax, (cm_mat, title, metrics) in zip(axes, [
    (cnn_cm, "Custom 6-Block CNN", cnn_metrics),
    (yolo_cm, "YOLOv8", yolo_metrics),
]):
    sns.heatmap(cm_mat, annot=True, fmt="d", cmap="Blues", xticklabels=short_names, yticklabels=short_names, ax=ax)
    ax.set_title(f"{title}\nAccuracy={metrics['accuracy']:.4f}  Scott's Pi={metrics['scotts_pi']:.4f}")
    ax.set_xlabel("Predicted Stage")
    ax.set_ylabel("True Stage")
plt.tight_layout()
plt.show()

print("Custom 6-Block CNN — Classification Report")
print(classification_report(y_true, cnn_preds, target_names=CLASS_NAMES))
print("YOLOv8 — Classification Report")
print(classification_report(y_true, yolo_preds, target_names=CLASS_NAMES))


## 13. Compare Candidate vs Incumbent `models/best.pt`
Before deciding whether to promote the new checkpoint, we run the **identical** evaluation procedure against the current `models/best.pt`, so both numbers come from the same test patients and the same metric code — no other difference.


In [ ]:
def evaluate_checkpoint(weights_path):
    yolo_model = YOLO(weights_path)
    preds = predict_and_score(yolo_model, test_df)
    return evaluate_predictions(y_true, preds, num_classes=4)

INCUMBENT_PATH = REPO_ROOT / "models" / "best.pt"

comparison_rows = {"candidate (yolo_candidate.pt)": yolo_metrics}
if INCUMBENT_PATH.exists():
    comparison_rows["incumbent (best.pt)"] = evaluate_checkpoint(INCUMBENT_PATH)
else:
    print(f"No incumbent checkpoint found at {INCUMBENT_PATH} — nothing to compare against yet.")
comparison_rows["documented target (YOLOv8)"] = DOCUMENTED_TARGETS["YOLOv8"]

comparison_df = pd.DataFrame(comparison_rows).T
comparison_df


## 14. Promote to `models/best.pt` — Manual Approval Required
**Do not run the cell below until you've reviewed the comparison table above.** `CONFIRM_OVERWRITE` defaults to `False`, so re-running the whole notebook never silently replaces the incumbent checkpoint. Flip it to `True` only after confirming the candidate reproduces or exceeds `models/best.pt`'s documented performance.


In [ ]:
CONFIRM_OVERWRITE = False  # Flip to True only after reviewing Section 13's comparison table

if CONFIRM_OVERWRITE:
    shutil.copy2(CANDIDATE_PATH, INCUMBENT_PATH)
    print(f"models/best.pt updated from {CANDIDATE_PATH}")
else:
    print(f"Skipped — models/best.pt left untouched. Candidate weights remain at {CANDIDATE_PATH}")
